In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from dotenv import load_dotenv
import janitor
from pathlib import Path
import openpyxl
import sqlalchemy as sa
import os
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Desactivar notacion cientifica
pd.set_option('display.float_format',  lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno 
load_dotenv()

print(f'✓ Librerías importadas correctamente')

# FASE 5: PRESELECCIÓN DE VARIABLES
## Agente A_05_SeleccionadorVariables

**Objetivo:** Reducir el número de variables manteniendo poder predictivo
- Método principal: RFECV con Regresión Logística L1
- Gestión de multicolinealidad: Desduplicación inteligente
- Output: Dataset preseleccionado con 22 features (vs 66 iniciales)

## PASO 1: Carga del Dataset Transformado

Cargar el dataframe transformado (67 features) desde la FASE 4

In [ ]:
# Cargar el dataframe transformado
import os
os.chdir('c:\\Users\\robin\\dev\\01_LEADSCORING')  # Cambiar al directorio del proyecto

ruta_datos = '02_datos/03_Entrenamiento/04_train_tablon_transformado.pkl'
df_modelo = pd.read_pickle(ruta_datos)

print(f'✓ Dataset cargado: {df_modelo.shape[0]} registros × {df_modelo.shape[1]} columnas')
print(f'\nPrimeras columnas: {list(df_modelo.columns[:5])}')
print(f'Última columna (target): {df_modelo.columns[-1]}')
print(f'\nTarget (compra) - Distribución:')
print(df_modelo['compra'].value_counts())
print(f'\nNulos totales: {df_modelo.isnull().sum().sum()}')

## PASO 2: Separación de Features y Target

In [ ]:
# Separar X (features) e y (target)
y = df_modelo['compra']
X = df_modelo.drop('compra', axis=1)

print(f'✓ Features (X): {X.shape[1]} variables')
print(f'✓ Target (y): {y.shape[0]} registros')
print(f'\nDistribución target:')
print(f'  - Clase 0: {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)')
print(f'  - Clase 1: {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)')

## PASO 3: RFECV con Regresión Logística L1

Método principal de preselección: elimina recursivamente features con baja importancia

In [ ]:
# Escalar features para Regresión Logística
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Configurar RFECV con Regresión Logística L1
modelo_l1 = LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=42, n_jobs=-1)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Ejecutar RFECV
rfecv = RFECV(
    estimator=modelo_l1,
    step=1,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)

print('🔄 Ejecutando RFECV con Regresión Logística L1...')
rfecv.fit(X_scaled, y)

print(f'\n✓ RFECV completado')
print(f'  - Número óptimo de features: {rfecv.n_features_}')
print(f'  - Reducción: {len(X.columns)} → {rfecv.n_features_} ({(1 - rfecv.n_features_/len(X.columns))*100:.1f}% reducción)')

## PASO 4: Variables Seleccionadas por RFECV

In [ ]:
# Obtener variables seleccionadas
variables_rfecv = X.columns[rfecv.support_].tolist()

print(f'✓ Variables seleccionadas por RFECV: {len(variables_rfecv)}')
print(f'\nVariables seleccionadas:')
for i, var in enumerate(variables_rfecv, 1):
    print(f'  {i:2d}. {var}')

# Almacenar ranking y scores
df_ranking_rfecv = pd.DataFrame({
    'variable': X.columns,
    'ranking': rfecv.ranking_,
    'seleccionada': rfecv.support_
}).sort_values('ranking')

print(f'\nVariables ELIMINADAS por RFECV: {(~rfecv.support_).sum()}')

## PASO 5: Desduplicación Inteligente

Eliminar variables altamente correlacionadas (preservando OHE, features temporales, etc.)

In [ ]:
# Calcular matriz de correlación sobre features seleccionados
X_seleccionado = X[variables_rfecv]
corr_matrix = X_seleccionado.corr().abs()

# Encontrar pares de variables altamente correlacionadas
umbral_correlacion = 0.9
pares_correlacionados = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > umbral_correlacion:
            pares_correlacionados.append({
                'var1': corr_matrix.columns[i],
                'var2': corr_matrix.columns[j],
                'correlacion': corr_matrix.iloc[i, j]
            })

if pares_correlacionados:
    print(f'⚠️  Encontrados {len(pares_correlacionados)} pares con correlación > {umbral_correlacion}:')
    for par in sorted(pares_correlacionados, key=lambda x: x['correlacion'], reverse=True)[:10]:
        print(f'  {par["var1"]} ↔ {par["var2"]}: {par["correlacion"]:.3f}')
else:
    print(f'✓ Sin variables altamente correlacionadas (umbral > {umbral_correlacion})')

# Eliminar variables más correlacionadas con usuario_nuevo
variables_a_eliminar = []
for par in pares_correlacionados:
    if 'usuario_nuevo' in par['var1'] or 'usuario_nuevo' in par['var2']:
        otra_var = par['var2'] if 'usuario_nuevo' in par['var1'] else par['var1']
        if otra_var not in variables_a_eliminar:
            variables_a_eliminar.append(otra_var)

print(f'\nVariables a eliminar (alta correlación con usuario_nuevo): {len(variables_a_eliminar)}')
if variables_a_eliminar:
    for var in variables_a_eliminar:
        print(f'  - {var}')

## PASO 6: Generación del Dataset Preseleccionado

In [ ]:
# Variables finales tras desduplicación
variables_preseleccionadas_final = [v for v in variables_rfecv if v not in variables_a_eliminar]

print(f'✓ Variables finales tras preselección: {len(variables_preseleccionadas_final)}')
print(f'\nResumen de reducciones:')
print(f'  - Iniciales: {len(X.columns)}')
print(f'  - Tras RFECV: {len(variables_rfecv)} ({(1 - len(variables_rfecv)/len(X.columns))*100:.1f}% reducción)')
print(f'  - Eliminadas por correlación: {len(variables_a_eliminar)}')
print(f'  - Finales: {len(variables_preseleccionadas_final)} ({(1 - len(variables_preseleccionadas_final)/len(X.columns))*100:.1f}% reducción total)')

# Crear dataset preseleccionado
df_preseleccionado = df_modelo[variables_preseleccionadas_final + ['compra']].copy()

print(f'\n✓ Dataset preseleccionado: {df_preseleccionado.shape[0]} registros × {df_preseleccionado.shape[1]} columnas')
print(f'\nNulos en dataset preseleccionado: {df_preseleccionado.isnull().sum().sum()}')

## PASO 7: Guardado de Artefactos

In [ ]:
# Guardar dataset preseleccionado
ruta_salida_datos = '02_datos/03_Entrenamiento/05_train_tablon_preseleccion.pkl'
df_preseleccionado.to_pickle(ruta_salida_datos)
print(f'✓ Dataset preseleccionado guardado: {ruta_salida_datos}')

# Guardar lista de variables
ruta_variables = '01_Documentos/Variables_preseleccionadas.txt'
os.makedirs(os.path.dirname(ruta_variables), exist_ok=True)
with open(ruta_variables, 'w') as f:
    for var in variables_preseleccionadas_final:
        f.write(f'{var}\n')
print(f'✓ Lista de variables guardada: {ruta_variables}')

## PASO 8: Resumen Final

In [ ]:
print('='*60)
print('📊 RESUMEN DE PRESELECCIÓN DE VARIABLES')
print('='*60)
print(f'\nMétodo utilizado: RFECV con Regresión Logística L1')
print(f'\nVariables iniciales: {len(X.columns)}')
print(f'Variables tras RFECV: {len(variables_rfecv)}')
print(f'Variables eliminadas por correlación: {len(variables_a_eliminar)}')
print(f'Variables finales: {len(variables_preseleccionadas_final)}')
print(f'\nReducción total: {len(X.columns)} → {len(variables_preseleccionadas_final)} ({(1 - len(variables_preseleccionadas_final)/len(X.columns))*100:.1f}%)')
print(f'\n✅ Dataset preseleccionado guardado en:')
print(f'   - Datos: {ruta_salida_datos}')
print(f'   - Variables: {ruta_variables}')
print('='*60)